<a href="https://colab.research.google.com/github/avantgarde2049/DUMMY_AGENTS_HF/blob/main/dummy_agent_library_groq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dummy Agent Library (Groq edition)

In this simple example, **we're going to code an Agent from scratch**.

This notebook is part of the <a href="https://www.hf.co/learn/agents-course">Hugging Face Agents Course</a>, adapted to run on **Groq's free inference API** instead of Hugging Face Inference Providers (whose free tier is ~$0.10/month and runs out fast).

Everything else is unchanged — Groq exposes the same `chat.completions.create` interface, so the agent logic below is identical to the original.

In [2]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.8 MB/s eta 0:00:00


## Groq free inference API

Groq runs open models (Llama, Qwen, GPT-OSS, Kimi) on their own hardware. The free tier needs **no credit card** — roughly 30 requests/minute and 14,400 requests/day, which is far more than this notebook will use.

To run this notebook, **you need a Groq API key** from https://console.groq.com/keys.
- On Google Colab: add it in the "Secrets" tab (key icon in the left sidebar), name it `GROQ_API_KEY`, and toggle notebook access on.
- Locally: set it as the `GROQ_API_KEY` environment variable.

Never paste the key directly into a cell — notebooks get committed to GitHub.

In [6]:
import os
from groq import Groq

## Get a key from https://console.groq.com/keys
## On Google Colab, add it in the "Secrets" tab (key icon, left sidebar) and call it "GROQ_API_KEY".
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except ImportError:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)

# Model IDs change as Groq rotates models in and out. If you get a 404 on the model,
# run the next cell to see what's currently live and pick another one.
MODEL = "meta-llama/llama-prompt-guard-2-22m"

In [7]:
# Optional: list the models currently available on your account
for m in sorted(client.models.list().data, key=lambda m: m.id):
    print(m.id)

allam-2-7b
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
qwen/qwen3.8-27b
whisper-large-v3
whisper-large-v3-turbo


We use the `chat.completions.create` method since it is a convenient and reliable way to apply chat templates:

In [9]:
output = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "The capital of France is"},
    ],
    stream=False,
    max_tokens=512,
)
print(output.choices[0].message.content)

0.0006226107361726463


This method is the RECOMMENDED one to use in order to ensure a **smooth transition between models** — it is the same call shape across Groq, OpenAI, Cerebras, OpenRouter and most other providers, so swapping backends is a one-line change.

## Dummy Agent

In the previous sections, we saw that the **core of an agent library is to append information in the system prompt**.

This system prompt is a bit more complex than the one we saw earlier, but it already contains:

1. **Information about the tools**
2. **Cycle instructions** (Thought → Action → Observation)

In [10]:
# This system prompt is a bit more complex and actually contains the function description already appended.
# Here we suppose that the textual description of the tools has already been appended.

SYSTEM_PROMPT = """Answer the following questions as best you can. You have access to the following tools:

get_weather: Get the current weather in a given location

The way you use the tools is by specifying a json blob.
Specifically, this json should have an `action` key (with the name of the tool to use) and an `action_input` key (with the input to the tool going here).

The only values that should be in the "action" field are:
get_weather: Get the current weather in a given location, args: {"location": {"type": "string"}}
example use :

{{
  "action": "get_weather",
  "action_input": {{"location": "New York"}}
}}


ALWAYS use the following format:

Question: the input question you must answer
Thought: you should always think about one action to take. Only one action at a time in this format:
Action:

$JSON_BLOB (inside markdown cell)

Observation: the result of the action. This Observation is unique, complete, and the source of truth.
... (this Thought/Action/Observation can repeat N times, you should take several steps when needed. The $JSON_BLOB must be formatted as markdown and only use a SINGLE action at a time.)

You must always end your output with the following format:

Thought: I now know the final answer
Final Answer: the final answer to the original input question

Now begin! Reminder to ALWAYS use the exact characters `Final Answer:` when you provide a definitive answer. """

We need to append the user instruction after the system prompt. This happens inside the `chat` method. We can see this process below:

In [11]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What's the weather in London?"},
]

print(messages)

[{'role': 'system', 'content': 'Answer the following questions as best you can. You have access to the following tools:\n\nget_weather: Get the current weather in a given location\n\nThe way you use the tools is by specifying a json blob.\nSpecifically, this json should have an `action` key (with the name of the tool to use) and an `action_input` key (with the input to the tool going here).\n\nThe only values that should be in the "action" field are:\nget_weather: Get the current weather in a given location, args: {"location": {"type": "string"}}\nexample use :\n\n{{\n  "action": "get_weather",\n  "action_input": {{"location": "New York"}}\n}}\n\n\nALWAYS use the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about one action to take. Only one action at a time in this format:\nAction:\n\n$JSON_BLOB (inside markdown cell)\n\nObservation: the result of the action. This Observation is unique, complete, and the source of truth.\n... (thi

Let's call the model!

In [13]:
# Use a conversational chat model instead of a prompt guard classification model
CHAT_MODEL = "qwen/qwen3.8-27b"  # or another chat/instruct model available on your Groq endpoint

output = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=messages,
    stream=False,
    max_tokens=200,
)
print(output.choices[0].message.content)

Question: What's the weather in London?
Thought: I need to retrieve the current weather for London using the get_weather tool.
Action:

```json
{
  "action": "get_weather",
  "action_input": {
    "location": "London"
  }
}
```


Do you see the issue?

> At this point, the model is hallucinating, because it's producing a fabricated "Observation" -- a response that it generates on its own rather than being the result of an actual function or tool call.
> To prevent this, we stop generating right before "Observation:".
> This allows us to manually run the function (e.g., `get_weather`) and then insert the real output as the Observation.

In [14]:
# The answer was hallucinated by the model. We need to stop to actually execute the function!
output = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=messages,
    max_tokens=150,
    stop=["Observation:"],  # Let's stop before any actual function is called
)

print(output.choices[0].message.content)

Thought: I need to get the current weather for London. I will use the get_weather tool.
Action:
```json
{
  "action": "get_weather",
  "action_input": {"location": "London"}
}
```


Much Better!

Let's now create a **dummy get weather function**. In a real situation you could call an API.

In [16]:
# Dummy function
def get_weather(location):
    return f"the weather in {location} is sunny with low temperatures. \n"

# Let's run it to define the function in the kernel namespace
get_weather('London')

'the weather in London is sunny with low temperatures. \n'

Let's concatenate the system prompt, the base prompt, the completion until function execution and the result of the function as an Observation and resume generation.

In [17]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What's the weather in London?"},
    {"role": "assistant", "content": output.choices[0].message.content + "Observation:\n" + get_weather('London')},
]

output = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=messages,
    stream=False,
    max_tokens=200,
)

print(output.choices[0].message.content)

Thought: I now know the final answer
Final Answer: The weather in London is sunny with low temperatures.


We learned how we can create Agents from scratch using Python code, and we **saw just how tedious that process can be**. Fortunately, many Agent libraries simplify this work by handling much of the heavy lifting for you.

Now, we're ready **to create our first real Agent** using the `smolagents` library.

---

### Note for later chapters

`smolagents` can also talk to Groq. Instead of `InferenceClientModel`, use the OpenAI-compatible model class:

```python
from smolagents import OpenAIServerModel

model = OpenAIServerModel(
    model_id="llama-3.3-70b-versatile",
    api_base="https://api.groq.com/openai/v1",
    api_key=GROQ_API_KEY,
)
```

Everything else in the course works unchanged.